# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and preprocess the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

**All `@id` fields are used to reference Record Sets, Fields, and Columns.**

In [ ]:
# List available Record Sets and their Fields by @id
print('Record Sets:')
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"  Record set name: {rs.name}")
    print(f"    @id: {rs.id}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      - {field.name}: @id {field.id}, dataType: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a pandas DataFrame for analysis.

All record sets and fields are referenced by their `@id` fields. See the printed overview above for the actual IDs.

In [ ]:
# Create a list of record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    # For each record set, extract the records into a DataFrame
    df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
    # Some record sets may be empty or have records=None, skip those
    if not df.empty:
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from record set @id: {rs_id}")
        print("Columns:", df.columns.tolist())
        display(df.head(2))

# Choose the main record set for further analysis (choose the one with actual data)
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]  # pick the first one with data
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering on a numeric field, normalization, grouping, etc.

All field references are given by their `@id`s.

In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id].copy()
    field_map = {field.id: field for field in dataset.metadata.record_set(main_record_set_id).fields}
    # Try to select a representative numeric field (e.g., 'MW', 'molecular_weight', etc. by @id)
    # Print all column names and types for inspection:
    print("All fields in chosen record set:")
    for f in dataset.metadata.record_set(main_record_set_id).fields:
        print(f"- {f.id} ({f.name}): {f.data_type}")

    # Find the first numeric field (schema:Float or schema:Integer)
    numeric_field_id = None
    for fid, field in field_map.items():
        if field.data_type in ["schema:Float", "schema:Integer", "schema:Number"]:
            if fid in df.columns:
                numeric_field_id = fid
                break
    
    if numeric_field_id is None:
        print("No numeric field found for analysis.")
    else:
        print(f"Using field @id '{numeric_field_id}' ({field_map[numeric_field_id].name}) for numeric analysis.")
        threshold = df[numeric_field_id].mean()  # Use mean as a sample threshold
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} records")
        display(filtered_df[[numeric_field_id]].head())

        # Normalization
        filtered_df[numeric_field_id + '_normalized'] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean() 
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

        # Try grouping by a categorical/text field
        group_field_id = None
        for fid, field in field_map.items():
            if field.data_type in ["schema:Text", "schema:String"] and fid in df.columns and fid != numeric_field_id:
                group_field_id = fid
                break
        if group_field_id:
            print(f"Grouping by field @id '{group_field_id}' ({field_map[group_field_id].name})")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print("Grouped DataFrame:")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
else:
    print("No record set with data available for EDA.")

## 5. Visualization
Visualize the distribution of the numeric field and relationship to the group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    fig, ax = plt.subplots(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, ax=ax)
    ax.set_title(f"Distribution of {numeric_field_id}")
    ax.set_xlabel(numeric_field_id)
    plt.show()
    
    if 'group_field_id' in locals() and group_field_id:
        # Show boxplot by group
        fig, ax = plt.subplots(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, ax=ax)
        ax.set_title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
- We demonstrated loading and basic exploration of a Croissant FAIR² dataset using `mlcroissant`.
- We explored available record sets and fields using their `@id` identifiers, examined numeric data distributions, and performed basic filtering/normalization/grouping.
- The use of `@id` for all data access ensures robust programmatic manipulation per the Croissant specification.

You can now proceed to advanced domain-specific analysis using the extracted DataFrames!